In [ ]:
# Standard library
import logging
import os
import warnings
import pathlib as pl

# Third-party libraries
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns

import tifffile
import timm
import torch
from PIL import Image
from tqdm.notebook import tqdm
from torchvision import transforms

# Warning filters
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# scGPT

In [ ]:
PATH_SCGPT_WEIGHTS = pl.Path('../../../Broad_SpatialFoundation/scGPT_model/')
PATH_UNI_WEIGHTS = pl.Path('../../../Broad_SpatialFoundation/UNI/pytorch_model.bin')

In [ ]:
import sys
sys.path.append('../../../Broad_SpatialFoundation/')

from sc_foundation_evals import cell_embeddings, scgpt_forward, data, model_output
from sc_foundation_evals.helpers.custom_logging import log

log.setLevel(logging.INFO)


In [ ]:
def run_embed_scGPT(
    dataset_path: str,
    model_dir: str,  # path to the pre-trained model, 3 files are expected: model_weights (best_model.pt), model args (args.json), and model vocab (vocab.json)
    output_dir: str,  # output_dir is the path to which the results should be saved
    n_hvg: int,
    gene_col: str = "index",  # in which column in adata.obs are gene names stored? if they are in index, the index will be copied to a column with this name
    layer_key: str = "X",  # where the raw counts are stored?
    log_norm: bool = False,  # are the values log_norm already?
    seed: int = 42,
    max_seq_len: int = 1200,  # maximum sequence of the input is controlled by max_seq_len, here I'm using the pretrained default
    batch_size: int = 32,  # batch_size depends on available GPU memory; should be a multiple of 8
    input_bins: int = 51,
    model_run: str = "pretrained",
    num_workers: int = 0,  # if you can use multithreading specify num_workers
) -> None:


    # create the model
    scgpt_model = scgpt_forward.scGPT_instance(
        saved_model_path=model_dir,
        model_run=model_run,
        batch_size=batch_size,
        save_dir=output_dir,
        num_workers=num_workers,
        explicit_save_dir=True,
    )

    # create config
    scgpt_model.create_configs(seed=seed, max_seq_len=max_seq_len, n_bins=input_bins)

    scgpt_model.load_pretrained_model()

    # This is to keep the model running if the amount of genes is small
    input_data = data.InputData(adata_dataset_path=dataset_path)
    vocab_list = scgpt_model.vocab.get_stoi().keys()

    adata = input_data.adata
    genes_in_vocab = adata.var_names.intersection(vocab_list)
    if len(genes_in_vocab) / len(adata.var_names) < 0.5:
        log.warning("Fewer than 50% of genes are found in the model vocab — continuing anyway.")
    
    adata._inplace_subset_var(genes_in_vocab)
    input_data.adata = adata

    input_data.preprocess_data(
        gene_vocab=vocab_list,
        model_type="scGPT",
        gene_col=gene_col,
        data_is_raw=not log_norm,
        counts_layer=layer_key,
        n_bins=input_bins,
        n_hvg=n_hvg,
    )

    scgpt_model.tokenize_data(
        data=input_data, input_layer_key="X_binned", include_zero_genes=False
    )

    scgpt_model.extract_embeddings(data=input_data)

    pd.DataFrame(
        input_data.adata.obsm["X_scGPT"],
        index=input_data.adata.obs["cell_id"] if "cell_id" in input_data.adata.obs.columns else input_data.adata.obs.index
    ).to_parquet(pl.Path(output_dir) / "scGPT.parquet")

In [ ]:
# change this to wherever you saved the data 
sample_name = 'CELLPOSE_Xenium_Ovarian-5k'
adata_path = "../../../Broad_SpatialFoundation/notebooks/notebooks_after_review/segmentation/xenium_cellpose.h5ad"

model_dir_GPT = PATH_SCGPT_WEIGHTS

In [ ]:
print(f'Starting for {sample_name}')
run_embed_scGPT(
    dataset_path=str(adata_path),
    model_dir=str(model_dir_GPT),
    output_dir="../../../Broad_SpatialFoundation/notebooks/notebooks_after_review/segmentation/embeddings",  
    n_hvg=1200,
    gene_col="index",
    layer_key="X",
    log_norm=False,
    seed=42,
    max_seq_len=1200,
    batch_size=16,
    input_bins=51,
    model_run="pretrained",
    num_workers=0,
)

# UNI

In [ ]:
def load_UNI_model(model_path: str, device: str = "cuda"):
    timm_kwargs = {
        'model_name': 'vit_giant_patch14_224',
        'img_size': 224,
        'patch_size': 14,
        'depth': 24,
        'num_heads': 24,
        'init_values': 1e-5,
        'embed_dim': 1536,
        'mlp_ratio': 2.66667 * 2,
        'num_classes': 0,
        'no_embed_class': True,
        'mlp_layer': timm.layers.SwiGLUPacked,
        'act_layer': torch.nn.SiLU,
        'reg_tokens': 8,
        'dynamic_img_size': True
    }

    model = timm.create_model(pretrained=False, **timm_kwargs)
    model.load_state_dict(torch.load(model_path, map_location="cpu"), strict=True)
    model.eval().to(device)

    transform = transforms.Compose([
        transforms.Resize(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=(0.485, 0.456, 0.406),
                             std=(0.229, 0.224, 0.225)),
    ])
    return model, transform


def embed_UNI(
    wsi, # whole-slide image as a NumPy array
    adata, # AnnData object; obs_names align with he_coords
    he_coords, # typically adata.obsm['spatial'], shape (n_spots, 2) with (x,y) pixel coords
    output_dir: str, # where to write embeddings, UNI.parquet
    model_path: str, # path to UNI weights
    batch_size: int = 128, # number of patches per forward pass
    device:str = "cuda", # "cuda" or "cpu"
):

    print('Load UNI model')
    model, transform = load_UNI_model(model_path, device)

    os.makedirs(output_dir, exist_ok=True)

    embeddings = []
    cell_ids = []
    
    batch_imgs = []
    batch_ids = []
    
    print(f"Embedding {len(he_coords)} image patches in batches of {batch_size}...")
    for cid, (x, y) in tqdm(zip(adata.obs_names, he_coords), total=len(adata)):
        x, y = int(x), int(y)
        x0, x1 = x - 128, x + 128
        y0, y1 = y - 128, y + 128
    
        pad_x0 = max(0, -x0)
        pad_x1 = max(0, x1 - wsi.shape[1])
        pad_y0 = max(0, -y0)
        pad_y1 = max(0, y1 - wsi.shape[0])
    
        patch = np.pad(
            wsi[max(0, y0):min(wsi.shape[0], y1), max(0, x0):min(wsi.shape[1], x1)],
            ((pad_y0, pad_y1), (pad_x0, pad_x1), (0, 0)),
            mode="constant"
        )
    
        if patch.shape[:2] != (256, 256):
            continue
    
        tensor_img = transform(Image.fromarray(patch))
        batch_imgs.append(tensor_img)
        batch_ids.append(cid)
    
        if len(batch_imgs) == batch_size:
            img_tensor = torch.stack(batch_imgs).to(device)
    
            with torch.inference_mode(), torch.autocast(device_type=device, dtype=torch.float16):
                batch_embs = model(img_tensor).to(torch.float16).cpu().numpy()
    
            embeddings.extend(batch_embs)
            cell_ids.extend(batch_ids)
            batch_imgs.clear()
            batch_ids.clear()
    
    # Final batch (if any)
    if batch_imgs:
        img_tensor = torch.stack(batch_imgs).to(device)
        with torch.inference_mode(), torch.autocast(device_type=device, dtype=torch.float16):
            batch_embs = model(img_tensor).to(torch.float16).cpu().numpy()
        embeddings.extend(batch_embs)
        cell_ids.extend(batch_ids)

    # Save embedding matrix
    df = pd.DataFrame(embeddings, index=cell_ids)
    df.to_parquet(f"{output_dir}/UNI.parquet")
    print(f"Saved {len(df)} embeddings to {output_dir}/UNI.parquet")    

In [ ]:
def _transform_x(aff_transf: pd.DataFrame, coords: np.ndarray) -> np.ndarray:
    """Why do we need this? The H&E image is not naturally aligned to the Xenium output. This can be done through the
    Xenium
    """

    inv_transf = np.linalg.inv(aff_transf)
    transformed_coords = (inv_transf @ np.vstack((coords.T, np.ones(len(coords))))).T[
        :, :-1
    ]

    return transformed_coords
    
# Alignment matrix from 10X
M = np.array([
    [0.010908748623278200,  1.2895248946320600, -721.007456942807],
    [-1.2895248946320600,  0.010908748623278200, 38642.677876412400],
    [0, 0, 1]
])

In [ ]:
source_image_path = '../../../Broad_SpatialFoundation/test_data/10X_Xenium_Ovarian_5k/Xenium_Prime_Ovarian_Cancer_FFPE_XRrun_he_image.ome.tif'

with tifffile.TiffFile(source_image_path) as tif:
    wsi = tif.series[0].asarray()

coords = adata.obsm["spatial_px"]
cell_names = adata.obs_names.to_numpy()

transformed_coords = _transform_x(aff_transf=M, coords=coords)
# Clip at 0 bc sometimes the transformation bugs a little bit, this should be minor though
# (ex: 1 of 150,000 cells had this in a dataset I am evaluating)
print(
    f"There are {((transformed_coords<0).sum(axis=1)>0).sum()} cells with negative coordinates, clipping at 0."
)
transformed_coords = transformed_coords.clip(0)


adata.obsm['spatial_he'] = transformed_coords

adata.obs['X_he'] = adata.obsm['spatial_he'][:,0]
adata.obs['Y_he'] = adata.obsm['spatial_he'][:,1]

In [ ]:
# set parameters to use with embed_UNI function
# specifically, output directory, cuda/cpu device, UNI model path,
# and h&e pixel coordinates
output_dir = '../../../Broad_SpatialFoundation/notebooks/notebooks_after_review/segmentation/embeddings'
device = "cuda" if torch.cuda.is_available() else "cpu"
model_path = PATH_UNI_WEIGHTS
he_coords = adata.obsm['spatial_he']

In [ ]:
embed_UNI(
    wsi,
    adata,
    he_coords,
    output_dir,
    model_path,
    batch_size = 512,
    device = device
)

# Virchow

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import timm
import tifffile

from PIL import Image
from tqdm import tqdm

from timm.layers import SwiGLUPacked
from timm.data import resolve_data_config
from timm.data.transforms_factory import create_transform


def load_Virchow2_model(device="cuda"):
    """
    Load Virchow2 model and official transforms.
    """

    model = timm.create_model(
        "hf-hub:paige-ai/Virchow2",
        pretrained=True,
        mlp_layer=SwiGLUPacked,
        act_layer=torch.nn.SiLU,
    )

    model.eval().to(device)

    transform = create_transform(
        **resolve_data_config(
            model.pretrained_cfg,
            model=model
        )
    )

    return model, transform


def load_wsi(path):
    """
    Load a whole-slide image into a NumPy array.
    """

    try:
        return tifffile.imread(path)

    except Exception:
        with tifffile.TiffFile(path) as tif:
            return tif.pages[0].asarray()


def embed_Virchow2(
    wsi,
    adata,
    he_coords,
    model=None,
    transform=None,
    output_dir=".",
    batch_size=128,
    device="cuda",
):

    print("=" * 80)
    print("Virchow2 embedding")
    print("=" * 80)

    print(f"WSI shape: {wsi.shape}")
    print(f"WSI dtype: {wsi.dtype}")
    print(f"Cells: {adata.n_obs:,}")
    print(f"Coords: {len(he_coords):,}")

    if model is None or transform is None:
        print("Loading Virchow2 model...")
        model, transform = load_Virchow2_model(device)

    os.makedirs(output_dir, exist_ok=True)

    embeddings = []
    cell_ids = []

    batch_imgs = []
    batch_ids = []

    device_type = torch.device(device).type

    # --------------------------------------------------
    # Debug counters
    # --------------------------------------------------

    total_cells = 0
    successful_cells = 0

    skipped_bad_shape = 0
    transform_failures = 0
    nan_coords = 0

    patch_shape_counts = {}

    x_vals = []
    y_vals = []

    print(
        f"\nEmbedding {len(he_coords):,} cells "
        f"(batch_size={batch_size})"
    )

    for cid, (x, y) in tqdm(
        zip(adata.obs_names, he_coords),
        total=len(adata),
        desc="Extracting patches"
    ):

        total_cells += 1

        # --------------------------
        # Coordinate sanity
        # --------------------------

        if np.isnan(x) or np.isnan(y):
            nan_coords += 1
            continue

        x_vals.append(x)
        y_vals.append(y)

        x = int(x)
        y = int(y)

        x0, x1 = x - 128, x + 128
        y0, y1 = y - 128, y + 128

        pad_x0 = max(0, -x0)
        pad_x1 = max(0, x1 - wsi.shape[1])

        pad_y0 = max(0, -y0)
        pad_y1 = max(0, y1 - wsi.shape[0])

        try:

            patch = np.pad(
                wsi[
                    max(0, y0):min(wsi.shape[0], y1),
                    max(0, x0):min(wsi.shape[1], x1)
                ],
                (
                    (pad_y0, pad_y1),
                    (pad_x0, pad_x1),
                    (0, 0),
                ),
                mode="constant"
            )

        except Exception as e:

            print(
                f"\nPatch extraction failed for "
                f"{cid}: {e}"
            )

            skipped_bad_shape += 1
            continue

        patch_shape_counts[patch.shape] = (
            patch_shape_counts.get(patch.shape, 0) + 1
        )

        # --------------------------
        # Force RGB
        # --------------------------

        if patch.ndim == 2:

            patch = np.stack(
                [patch] * 3,
                axis=-1
            )

        elif patch.shape[-1] > 3:

            patch = patch[..., :3]

        if patch.shape[:2] != (256,256):

            skipped_bad_shape += 1

            if skipped_bad_shape < 20:

                print(
                    f"\nBAD PATCH {cid}",
                    f"\nx={x}, y={y}",
                    f"\nx0={x0}, x1={x1}",
                    f"\ny0={y0}, y1={y1}",
                    f"\nraw slice shape={wsi[max(0,y0):min(wsi.shape[0],y1), max(0,x0):min(wsi.shape[1],x1)].shape}",
                    f"\nfinal patch shape={patch.shape}",
                )

                print(
                    f"\nBad patch shape "
                    f"{patch.shape} "
                    f"for cell {cid}"
                )

            continue

        try:

            tensor_img = transform(
                Image.fromarray(
                    patch.astype(np.uint8)
                )
            )

        except Exception as e:

            transform_failures += 1

            if transform_failures < 20:

                print(
                    f"\nTransform failed "
                    f"for {cid}: {e}"
                )

            continue

        batch_imgs.append(tensor_img)
        batch_ids.append(cid)

        # --------------------------
        # Inference
        # --------------------------

        if len(batch_imgs) == batch_size:

            img_tensor = torch.stack(
                batch_imgs
            ).to(device)

            with torch.inference_mode(), torch.autocast(
                device_type=device_type,
                dtype=torch.float16,
            ):

                output = model(img_tensor)

                class_token = output[:, 0]
                patch_tokens = output[:, 5:]

                batch_embs = torch.cat(
                    [
                        class_token,
                        patch_tokens.mean(1)
                    ],
                    dim=-1
                )

                batch_embs = (
                    batch_embs
                    .cpu()
                    .numpy()
                )

            embeddings.extend(batch_embs)
            cell_ids.extend(batch_ids)

            successful_cells += len(batch_ids)

            batch_imgs.clear()
            batch_ids.clear()

    # --------------------------------------------------
    # Final batch
    # --------------------------------------------------

    if batch_imgs:

        img_tensor = torch.stack(
            batch_imgs
        ).to(device)

        with torch.inference_mode(), torch.autocast(
            device_type=device_type,
            dtype=torch.float16,
        ):

            output = model(img_tensor)

            class_token = output[:, 0]
            patch_tokens = output[:, 5:]

            batch_embs = torch.cat(
                [
                    class_token,
                    patch_tokens.mean(1)
                ],
                dim=-1
            )

            batch_embs = (
                batch_embs
                .cpu()
                .numpy()
            )

        embeddings.extend(batch_embs)
        cell_ids.extend(batch_ids)

        successful_cells += len(batch_ids)

    # --------------------------------------------------
    # Debug summary
    # --------------------------------------------------

    print("\n" + "=" * 80)
    print("SUMMARY")
    print("=" * 80)

    print(f"Total cells:           {total_cells:,}")
    print(f"Embedded cells:        {successful_cells:,}")
    print(f"Missing cells:         {total_cells-successful_cells:,}")

    print(f"Bad patch shape:       {skipped_bad_shape:,}")
    print(f"Transform failures:    {transform_failures:,}")
    print(f"NaN coordinates:       {nan_coords:,}")

    print("\nCoordinate ranges")

    print(
        f"x: {np.min(x_vals):.1f} "
        f"→ {np.max(x_vals):.1f}"
    )

    print(
        f"y: {np.min(y_vals):.1f} "
        f"→ {np.max(y_vals):.1f}"
    )

    print("\nTop patch shapes:")

    for shape, count in sorted(
        patch_shape_counts.items(),
        key=lambda x: x[1],
        reverse=True,
    )[:20]:

        print(f"{shape}: {count:,}")

    # --------------------------------------------------
    # Save
    # --------------------------------------------------

    df = pd.DataFrame(
        embeddings,
        index=cell_ids,
    )

    outfile = os.path.join(
        output_dir,
        "Virchow2.parquet",
    )

    df.to_parquet(outfile)

    print(
        f"\nSaved {len(df):,} embeddings"
        f"\n{outfile}"
    )

    return df

In [ ]:
device = "cuda:4" if torch.cuda.is_available() else "cpu"

adata = sc.read_h5ad('../../../Broad_SpatialFoundation/notebooks/notebooks_after_review/segmentation/xenium_cellpose.h5ad')

In [ ]:
source_image_path = "../../../Broad_SpatialFoundation/test_data/10X_Xenium_Ovarian_5k/Xenium_Prime_Ovarian_Cancer_FFPE_XRrun_he_image.ome.tif"

with tifffile.TiffFile(source_image_path) as tif:
    wsi = tif.series[0].asarray()

In [ ]:
def _transform_x(aff_transf: pd.DataFrame, coords: np.ndarray) -> np.ndarray:
    """Why do we need this? The H&E image is not naturally aligned to the Xenium output. This can be done through the
    Xenium
    """

    inv_transf = np.linalg.inv(aff_transf)
    transformed_coords = (inv_transf @ np.vstack((coords.T, np.ones(len(coords))))).T[
        :, :-1
    ]

    return transformed_coords
    
# Alignment matrix from 10X
M = np.array([
    [0.010908748623278200,  1.2895248946320600, -721.007456942807],
    [-1.2895248946320600,  0.010908748623278200, 38642.677876412400],
    [0, 0, 1]
])

In [ ]:
coords = adata.obsm["spatial_px"]
cell_names = adata.obs_names.to_numpy()

transformed_coords = _transform_x(aff_transf=M, coords=coords)
# Clip at 0 bc sometimes the transformation bugs a little bit, this should be minor though
# (ex: 1 of 150,000 cells had this in a dataset I am evaluating)
print(
    f"There are {((transformed_coords<0).sum(axis=1)>0).sum()} cells with negative coordinates, clipping at 0."
)
transformed_coords = transformed_coords.clip(0)


adata.obsm['spatial_he'] = transformed_coords

adata.obs['X_he'] = adata.obsm['spatial_he'][:,0]
adata.obs['Y_he'] = adata.obsm['spatial_he'][:,1]

In [ ]:
model, transform = load_Virchow2_model(device)

In [ ]:
he_coords = adata.obsm["spatial_he"]

embed_Virchow2(
    wsi=wsi,
    adata=adata,
    he_coords=he_coords,
    model=model,
    transform=transform,
    output_dir="../../../Broad_SpatialFoundation/notebooks/notebooks_after_review/segmentation/embeddings/",
    batch_size=512,
    device=device,
)

# Nicheformer

In [ ]:
import gc
import gzip
import logging
import pathlib as pl
from datetime import datetime

import anndata as ad
import nicheformer
import numpy as np
import pandas as pd
import scanpy as sc
import torch
import tqdm

from torch.utils.data import DataLoader


In [ ]:
# ============================================================
# Paths
# ============================================================

BASE_DIR = pl.Path(
    "../../../Broad_SpatialFoundation/test_data/"
)

MODEL_PATH = (
    "../../../Broad_SpatialFoundation/"
    "nicheformer_model/nicheformer.ckpt"
)

VOCAB_PATH = (
    "../../../Broad_SpatialFoundation/"
    "nicheformer_model/model.h5ad"
)

TECH_MEAN_PATH = (
    "../../../Broad_SpatialFoundation/"
    "nicheformer_model/xenium_mean_script.npy"
)

GTF_PATH = (
    "../../../Broad_SpatialFoundation/"
    "gencode.v48.basic.annotation.gtf.gz"
)

In [ ]:

# ============================================================
# Config
# ============================================================

CONFIG = {
    "batch_size": 32,
    "max_seq_len": 1500,
    "aux_tokens": 30,
    "chunk_size": 1000,
    "num_workers": 0,
    "embedding_layer": -1,
}

# ============================================================
# Logging
# ============================================================

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

log_file = BASE_DIR / f"nicheformer_batch_{timestamp}.log"

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[
        logging.FileHandler(log_file),
        logging.StreamHandler(),
    ],
)

logging.info("Starting Nicheformer batch embedding")

# ============================================================
# Utilities
# ============================================================

def set_seed(seed=42):

    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# ============================================================
# Build symbol -> Ensembl mapping
# ============================================================

def build_symbol_to_ensembl_map(gtf_path):

    logging.info("Parsing GTF")

    records = []

    with gzip.open(gtf_path, "rt") as f:

        for line in f:

            if line.startswith("#"):
                continue

            fields = line.strip().split("\t")

            if fields[2] != "gene":
                continue

            attrs = fields[8]

            attr_dict = {}

            for item in attrs.split(";"):

                item = item.strip()

                if item == "":
                    continue

                key, value = item.split(" ", 1)

                attr_dict[key] = value.strip('"')

            gene_id = attr_dict.get("gene_id")
            gene_name = attr_dict.get("gene_name")

            if gene_id and gene_name:

                gene_id = gene_id.split(".")[0]

                records.append(
                    (
                        gene_name.upper(),
                        gene_id,
                    )
                )

    mapping_df = pd.DataFrame(
        records,
        columns=["gene_symbol", "ensembl_id"],
    ).drop_duplicates()

    symbol_to_ens = mapping_df.set_index(
        "gene_symbol"
    )["ensembl_id"].to_dict()

    logging.info(
        f"Parsed {len(symbol_to_ens)} mappings"
    )

    return symbol_to_ens


# ============================================================
# Remove Xenium controls
# ============================================================

def remove_control_probes(adata):

    mask = ~adata.var_names.str.upper().str.startswith(
        (
            "BLANK_",
            "NEGCONTROLCODEWORD",
            "NEGCONTROLPROBE",
            "ANTISENSE_",
        )
    )

    return adata[:, mask].copy()


# ============================================================
# Map genes to Ensembl
# ============================================================

def map_genes_to_ensembl(
    adata,
    symbol_to_ens,
):

    adata.var_names = (
        adata.var_names
        .str.strip()
        .str.upper()
    )

    adata.var_names_make_unique()

    adata.var["ensembl_id"] = [
        symbol_to_ens.get(g, None)
        for g in adata.var_names
    ]

    mapped = adata.var["ensembl_id"].notnull().sum()

    logging.info(
        f"Mapped genes: {mapped}/{adata.n_vars}"
    )

    adata = adata[
        :,
        adata.var["ensembl_id"].notnull()
    ].copy()

    adata.var_names = (
        adata.var["ensembl_id"]
        .astype(str)
    )

    adata.var_names_make_unique()

    return adata


# ============================================================
# Align to vocab
# ============================================================

def align_to_vocab(
    adata,
    vocab,
):

    vocab = vocab[
        :,
        vocab.var_names.str.startswith("ENSG")
    ].copy()

    common_genes = vocab.var_names.intersection(
        adata.var_names
    )

    logging.info(
        f"Common genes: {len(common_genes)}"
    )

    adata = adata[:, common_genes].copy()

    ordered_genes = [
        g for g in vocab.var_names
        if g in adata.var_names
    ]

    adata = adata[:, ordered_genes].copy()

    return adata, ordered_genes, vocab


# ============================================================
# Align technology mean
# ============================================================

def align_technology_mean(
    ordered_genes,
    vocab,
    tech_mean_path,
):

    technology_mean_full = np.load(
        tech_mean_path
    )

    tech_mean_map = dict(
        zip(vocab.var_names, technology_mean_full)
    )

    technology_mean = np.array([
        tech_mean_map[g]
        for g in ordered_genes
    ])

    return technology_mean.astype(np.float32)


# ============================================================
# Add metadata
# ============================================================

def add_metadata(adata):

    adata.obs["modality"] = 4
    adata.obs["species"] = 5
    adata.obs["assay"] = 9

    if "nicheformer_split" not in adata.obs.columns:

        adata.obs["nicheformer_split"] = "train"

    return adata


# ============================================================
# Main embedding function
# ============================================================

def run_embed_nicheformer(
    dataset_path,
    output_dir,
    symbol_to_ens,
):

    logging.info(f"Loading {dataset_path}")

    adata = ad.read_h5ad(dataset_path)

    logging.info(f"Original shape: {adata.shape}")

    if "cell_id" in adata.obs:

        cell_ids = (
            adata.obs["cell_id"]
            .astype(str)
        )

    else:

        cell_ids = (
            adata.obs_names
            .astype(str)
        )

    # --------------------------------------------------------
    # Remove controls
    # --------------------------------------------------------

    adata = remove_control_probes(
        adata
    )

    logging.info(
        f"After removing controls: {adata.shape}"
    )

    # --------------------------------------------------------
    # Map to Ensembl
    # --------------------------------------------------------

    adata = map_genes_to_ensembl(
        adata,
        symbol_to_ens,
    )

    logging.info(
        f"After mapping: {adata.shape}"
    )

    # --------------------------------------------------------
    # Load vocab
    # --------------------------------------------------------

    vocab = sc.read_h5ad(
        VOCAB_PATH
    )

    # --------------------------------------------------------
    # Align genes
    # --------------------------------------------------------

    adata, ordered_genes, vocab = align_to_vocab(
        adata,
        vocab,
    )

    logging.info(
        f"Final aligned shape: {adata.shape}"
    )

    # --------------------------------------------------------
    # Align tech mean
    # --------------------------------------------------------

    technology_mean = align_technology_mean(
        ordered_genes,
        vocab,
        TECH_MEAN_PATH,
    )

    # --------------------------------------------------------
    # Convert counts to float32
    # --------------------------------------------------------

    adata.X = adata.X.astype(
        np.float32
    )

    # --------------------------------------------------------
    # Metadata
    # --------------------------------------------------------

    adata = add_metadata(
        adata
    )

    # --------------------------------------------------------
    # Dataset
    # --------------------------------------------------------

    dataset = nicheformer.data.NicheformerDataset(
        adata=adata,
        technology_mean=technology_mean,
        split="train",
        max_seq_len=CONFIG["max_seq_len"],
        aux_tokens=CONFIG["aux_tokens"],
        chunk_size=CONFIG["chunk_size"],
        metadata_fields={
            "obs": [
                "modality",
                "species",
                "assay",
            ]
        },
    )

    logging.info(
        f"Token shape: {dataset.tokens.shape}"
    )

    # --------------------------------------------------------
    # Dataloader
    # --------------------------------------------------------

    dataloader = DataLoader(
        dataset,
        batch_size=CONFIG["batch_size"],
        shuffle=False,
        num_workers=CONFIG["num_workers"],
        pin_memory=True,
    )

    # --------------------------------------------------------
    # Load model
    # --------------------------------------------------------

    model = (
        nicheformer.models.Nicheformer
        .load_from_checkpoint(
            checkpoint_path=MODEL_PATH,
            strict=False,
        )
    )

    model.eval()

    device = torch.device(
        "cuda"
        if torch.cuda.is_available()
        else "cpu"
    )

    model = model.to(device)

    # --------------------------------------------------------
    # Generate embeddings
    # --------------------------------------------------------

    embeddings = []

    with torch.no_grad():

        for batch in tqdm.tqdm(dataloader):

            batch = {
                k: v.to(device)
                if isinstance(v, torch.Tensor)
                else v
                for k, v in batch.items()
            }

            emb = model.get_embeddings(
                batch=batch,
                layer=CONFIG["embedding_layer"],
            )

            embeddings.append(
                emb.cpu().numpy()
            )

            gc.collect()

    embeddings = np.concatenate(
        embeddings,
        axis=0,
    )

    # --------------------------------------------------------
    # Save parquet
    # --------------------------------------------------------

    outfile = (
        pl.Path(output_dir)
        / "nicheformer.parquet"
    )

    pd.DataFrame(
        embeddings,
        index=cell_ids,
    ).to_parquet(outfile)

    logging.info(
        f"Saved embeddings to {outfile}"
    )




In [ ]:
BASE_DIR = pl.Path('../../../Broad_SpatialFoundation/notebooks/notebooks_after_review/segmentation/')

In [ ]:
set_seed(42)

symbol_to_ens = (
    build_symbol_to_ensembl_map(
        GTF_PATH
    )
)

sample_dir = BASE_DIR 

sample_name = 'Cellpose_OVCA'

logging.info(
    f"Processing {sample_name}"
)

adata_path = (
    sample_dir / "xenium_cellpose.h5ad"
)

output_dir = (
    sample_dir / "embeddings"
)

output_dir.mkdir(
    exist_ok=True
)

outfile = (
    output_dir
    / "nicheformer.parquet"
)

# ----------------------------------------------------
# Run embedding
# ----------------------------------------------------

run_embed_nicheformer(
        dataset_path=adata_path,
        output_dir=output_dir,
        symbol_to_ens=symbol_to_ens,
    )

logging.info(
    f"Finished {sample_name}"
)

# Now SpatialFusion

In [ ]:
from spatialfusion.embed.embed import AEInputs, run_full_embedding

In [ ]:
output_dir = pl.Path('../../../Broad_SpatialFoundation/notebooks/notebooks_after_review/segmentation/')
sample_name = 'CELLPOSE_Xenium_Ovarian-5k'

adata.obs = pd.concat([adata.obs, pd.DataFrame(adata.obsm['spatial_px'], index=adata.obs_names, columns=['X_coord','Y_coord'])],axis=1)
adata.obs["sample_id"] = sample_name

In [ ]:
uni_df = pd.read_parquet(pl.Path(output_dir) / 'embeddings' / 'UNI.parquet')
scgpt_df = pd.read_parquet(pl.Path(output_dir) / 'embeddings' / 'scGPT.parquet')
virchow_df = pd.read_parquet(pl.Path(output_dir) / 'embeddings' / 'Virchow2.parquet')
nicheformer_df = pd.read_parquet(pl.Path(output_dir) / 'embeddings' / 'nicheformer.parquet')

In [ ]:
adata.obs_names = "cell_"+adata.obs_names

In [ ]:
uni_df.index ="cell_"+uni_df.index.astype(str)
scgpt_df.index ="cell_"+scgpt_df.index.astype(str)
virchow_df.index ="cell_"+virchow_df.index.astype(str)
nicheformer_df.index ="cell_"+nicheformer_df.index.astype(str)

# Embedding UNI2+scGPT

In [ ]:
ae_inputs_by_sample = {
    sample_name: AEInputs(adata=adata, z_he=uni_df, z_rna=scgpt_df),
}

In [ ]:
# this uses the average version
embeddings_df = run_full_embedding(
    ae_inputs_by_sample=ae_inputs_by_sample,
    ae_model_path='../../../Broad_SpatialFoundation/checkpoint_dir_ae/paired_model_6c22d731.pt',
    gcn_model_path='../../../Broad_SpatialFoundation/checkpoint_dir_gcn/gcn_20250828-123835_e926ee8d/model.pt',
    device="cuda:1",
    combine_mode="average",
    spatial_key='spatial_px',
    celltype_key='major_celltype',
    save_ae_dir=None,  # optional
)

In [ ]:
out_path = output_dir /  "uni_scgpt_avg.parquet"
embeddings_df.to_parquet(out_path)

# Embedding UNI2+Nicheformer

In [ ]:
ae_inputs_by_sample = {
    sample_name: AEInputs(adata=adata, z_he=uni_df, z_rna=nicheformer_df),
}

In [ ]:
# this uses the average version
embeddings_df = run_full_embedding(
    ae_inputs_by_sample=ae_inputs_by_sample,
    ae_model_path='../../../SpatialFusion/results/ae_encoder_sweep/uni_nicheformer_full_20260602-051319_a4b53581/model.pt',
    gcn_model_path='../../../SpatialFusion/results/gcn_fusion_sweep/uni_nicheformer_full_gcn_avg_reg_cls_80d67272/model.pt',
    device="cuda:1",
    combine_mode="average",
    spatial_key='spatial_px',
    celltype_key='major_celltype',
    save_ae_dir=None,  # optional
)

In [ ]:
out_path = output_dir /  "uni_nicheformer_avg.parquet"
embeddings_df.to_parquet(out_path)

# Embedding Virchow2+scGPT

In [ ]:
ae_inputs_by_sample = {
    sample_name: AEInputs(adata=adata, z_he=virchow_df, z_rna=scgpt_df),
}

In [ ]:
# this uses the average version
embeddings_df = run_full_embedding(
    ae_inputs_by_sample=ae_inputs_by_sample,
    ae_model_path='../../../SpatialFusion/results/ae_encoder_sweep/virchow_scgpt_full_20260602-065430_e436487d/model.pt',
    gcn_model_path='../../../SpatialFusion/results/gcn_fusion_sweep/virchow_scgpt_full_gcn_avg_reg_cls_6c8c309d/model.pt',
    device="cuda:1",
    combine_mode="average",
    spatial_key='spatial_px',
    celltype_key='major_celltype',
    save_ae_dir=None,  # optional
)

In [ ]:
out_path = output_dir /  "virchow_scgpt_avg.parquet"
embeddings_df.to_parquet(out_path)

# Embedding Virchow2+Nicheformer

In [ ]:
ae_inputs_by_sample = {
    sample_name: AEInputs(adata=adata, z_he=virchow_df, z_rna=nicheformer_df),
}

In [ ]:
# this uses the average version
embeddings_df = run_full_embedding(
    ae_inputs_by_sample=ae_inputs_by_sample,
    ae_model_path='../../../SpatialFusion/results/ae_encoder_sweep/virchow_nicheformer_full_20260602-084626_531b291c/model.pt',
    gcn_model_path='../../../SpatialFusion/results/gcn_fusion_sweep/virchow_nicheformer_full_gcn_avg_reg_cls_05276acb/model.pt',
    device="cuda:1",
    combine_mode="average",
    spatial_key='spatial_px',
    celltype_key='major_celltype',
    save_ae_dir=None,  # optional
)

In [ ]:
out_path = output_dir /  "virchow_nicheformer_avg.parquet"
embeddings_df.to_parquet(out_path)